# System Prompt Ablation: ECE Debugging

Investigating why **conservative** sysprompt improves ECE without changing accuracy or uncertainty.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.environ['OPENESTIMATE_ROOT'] = '/home/ubuntu/alana/openestimate/'
from utils import load_experiment_results

# Load ablation data
datasets = ['nhanes', 'pitchbook', 'glassdoor']
results = {d: load_experiment_results(d, 'ablations') for d in datasets}

In [ ]:
# Filter to o4-mini, direct protocol, medium temp - compare sysprompts
def filter_sysprompt(df):
    return df[(df['model'] == 'o4-mini') & 
              (df['temperature'] == 'medium') & 
              (df['elicitation_protocol'] == 'direct') &
              (df['sysprompt_type'].isin(['base', 'conservative']))].copy()

filtered = {d: filter_sysprompt(r) for d, r in results.items()}
for d, df in filtered.items():
    print(f"{d}: {df['sysprompt_type'].value_counts().to_dict()}")

In [ ]:
# Compare ECE, error, std by sysprompt
def compute_metrics(df):
    q_counts = df['quartile_of_gt'].value_counts(normalize=True) * 100
    ece = sum(abs(q_counts.get(q, 0) - 25) for q in [1,2,3,4]) / 4
    return {'ece': ece, 'error': df['error_ratio_mean'].mean(), 
            'std': df['std_ratio'].mean(),
            'q1': q_counts.get(1,0), 'q2': q_counts.get(2,0),
            'q3': q_counts.get(3,0), 'q4': q_counts.get(4,0)}

for d, df in filtered.items():
    print(f"\n{d.upper()}:")
    for sp in ['base', 'conservative']:
        m = compute_metrics(df[df['sysprompt_type'] == sp])
        print(f"  {sp}: ECE={m['ece']:.1f}, Error={m['error']:.2f}, Std={m['std']:.2f}")
        print(f"    Quartiles: Q1={m['q1']:.1f}% Q2={m['q2']:.1f}% Q3={m['q3']:.1f}% Q4={m['q4']:.1f}%")

In [ ]:
# Visualize quartile distributions
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for idx, (d, df) in enumerate(filtered.items()):
    ax = axes[idx]
    for i, sp in enumerate(['base', 'conservative']):
        sub = df[df['sysprompt_type'] == sp]
        pcts = [(sub['quartile_of_gt'] == q).mean()*100 for q in [1,2,3,4]]
        ax.bar(np.arange(4) + i*0.35, pcts, 0.35, label=sp)
    ax.axhline(25, color='red', ls='--', label='ideal')
    ax.set_xticks(np.arange(4) + 0.175)
    ax.set_xticklabels(['Q1','Q2','Q3','Q4'])
    ax.set_title(d.upper())
    ax.set_ylim(0, 50)
    if idx == 2: ax.legend()
plt.tight_layout()
plt.savefig('analysis_results/sysprompt_quartile_comparison.png', dpi=150)
plt.show()

In [ ]:
# Find variables where quartile changes but error doesn't
for d, df in filtered.items():
    base = df[df['sysprompt_type'] == 'base']
    cons = df[df['sysprompt_type'] == 'conservative']
    
    shifts = []
    for var in base['variable'].unique():
        b, c = base[base['variable']==var], cons[cons['variable']==var]
        if len(b) == 0 or len(c) == 0: continue
        shifts.append({
            'var': var[:40],
            'q_shift': c['quartile_of_gt'].mean() - b['quartile_of_gt'].mean(),
            'err_change': (c['abs_error_from_mean'].mean() - b['abs_error_from_mean'].mean()) / b['abs_error_from_mean'].mean() * 100
        })
    
    shifts_df = pd.DataFrame(shifts).sort_values('q_shift', ascending=False)
    print(f"\n{d.upper()} - Top quartile shifts with small error change:")
    print(shifts_df[abs(shifts_df['err_change']) < 20].head(5).to_string(index=False))

In [ ]:
# Overconfidence analysis: Q1+Q4 should be 50% for perfect calibration
print("Overconfidence (Q1+Q4 - 50%, positive = overconfident):")
for d, df in filtered.items():
    print(f"\n{d.upper()}:")
    for sp in ['base', 'conservative']:
        sub = df[df['sysprompt_type'] == sp]
        extreme = ((sub['quartile_of_gt'] == 1) | (sub['quartile_of_gt'] == 4)).mean() * 100
        print(f"  {sp}: {extreme:.1f}% in extremes ({extreme-50:+.1f}% overconfidence)")